# positive_control_lr_probe

Sweeps three learning rates for the memorization positive control, 20 steps each, to select a
rate before committing the full run.

The control trains and scores on the same 302 rows (`screen.jsonl`), so the question is whether
the optimizer moves the adapter at all, not whether it generalizes. Rate selection is on
training loss and adapter movement. Validation loss is not used: it does not track the GPT-4.1
task metric on this pipeline, and there is no held-out set here by construction.

Shipped `learning_rate` is 8e-07. The grid is 1e-05 / 1e-04 / 3e-04, spanning two orders of
magnitude above it.

`warmup_t` is 5, not the shipped 500: at 500 every step of a 20-step probe sits inside warmup at
roughly 2% of the configured rate and measures nothing.

Rank 8, matching the arm whose result the control underwrites. Every other config key is
inherited from `options/ft_lora.yml` unchanged.

Peak VRAM 45.4 GB, measured by `lora_variant_timing`, so an 80 GB A100 is required. A 40 GB
A100 is sufficient for generation but not for training. Roughly 25 minutes of sweep after setup.

## 1. Repo and dependencies

In [1]:
import os, subprocess, sys, json, shutil

REPO = "/content/OmniGen2"
COMMIT = "18e6f9d5271b517fcb32e999f10df943ae9b8f20"

if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "https://github.com/VectorSpaceLab/OmniGen2.git", REPO], check=True)
    print("cloned")
else:
    print("repo already present")

subprocess.run(["git", "-C", REPO, "checkout", "-q", COMMIT], check=True)
print("commit:", subprocess.run(["git", "-C", REPO, "rev-parse", "HEAD"],
                                capture_output=True, text=True).stdout.strip())

# requirements.txt omits torchdiffeq, imported by omnigen2/transport/integrators.py
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], cwd=REPO, check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torchdiffeq"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U",
                "huggingface_hub[hf_xet]>=0.30.0,<1.0"], check=True)

# peft raises rather than returning False when torchao predates 0.16.0; Colab ships 0.10.0.
# torchao is unused by the bf16 LoRA path.
_ao = subprocess.run([sys.executable, "-c",
                      "from peft.import_utils import is_torchao_available; is_torchao_available()"],
                     capture_output=True, text=True)
if _ao.returncode != 0:
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"], check=True)
    print("torchao removed")

_probe = ("from peft import LoraConfig;"
          "from peft.import_utils import is_torchao_available; is_torchao_available();"
          "from omnigen2.transport import create_transport;"
          "from omnigen2.dataset.omnigen2_train_dataset import OmniGen2TrainDataset;"
          "print('train.py imports OK')")
chk = subprocess.run([sys.executable, "-c", _probe], cwd=REPO, capture_output=True, text=True)
print(chk.stdout.strip() or chk.stderr.strip()[-1000:])
assert chk.returncode == 0, "train.py imports are broken"

sys.path.insert(0, REPO)
os.chdir(REPO)

import torch
print("gpu :", torch.cuda.get_device_name(0),
      "|", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")
assert torch.cuda.get_device_properties(0).total_memory / 1e9 > 60, \
    "peak usage is 45.4 GB; an 80 GB A100 is required"

cloned
commit: 18e6f9d5271b517fcb32e999f10df943ae9b8f20
torchao removed
train.py imports OK
gpu : NVIDIA A100-SXM4-80GB | 85.1 GB


## 2. Drive and images

Extracted straight from Drive rather than copied locally first: holding a 48 GB archive and
48 GB of extracted files at once overflows the Colab disk when model weights are also cached.

In [2]:
from google.colab import drive, userdata

drive.mount("/content/drive")
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

DRIVE = "/content/drive/MyDrive/Colab Notebooks/My Master's Thesis/Data"
IMGS, EXPECTED = "/content/imgs", 37063

def n_files(d):
    return sum(len(f) for _, _, f in os.walk(d)) if os.path.isdir(d) else 0

if n_files(IMGS) < EXPECTED:
    free_gb = shutil.disk_usage("/content").free / 1e9
    print(f"free before restore: {free_gb:.0f} GB")
    assert free_gb > 70, f"only {free_gb:.0f} GB free; the image set needs ~50 GB plus model cache"
    os.makedirs(IMGS, exist_ok=True)
    # extracted straight from Drive: copying the tar locally first would hold 48 GB of archive
    # and 48 GB of extracted files at the same time, which overflows the Colab disk
    subprocess.run(f'tar -xf "{DRIVE}/imgs_subset.tar" -C {IMGS}', shell=True, check=True)
    assert n_files(IMGS) >= EXPECTED, f"restore incomplete: {n_files(IMGS)} of {EXPECTED}"
print("images:", n_files(IMGS))

for f in ["screen.jsonl", "test.jsonl", "val.jsonl"]:
    if not os.path.exists(f"/content/{f}"):
        subprocess.run(["cp", f"{DRIVE}/{f}", "/content/"], check=True)

print(subprocess.run(["df", "-h", "/content"], capture_output=True, text=True).stdout.strip().split("\n")[-1])

Mounted at /content/drive
free before restore: 193 GB
images: 37063
overlay         236G  146G   90G  62% /


## 3. Memorization set

`screen.jsonl` verbatim: 302 rows over 62 targets, carved from train by target at seed 42 and
listed in `screen_exclude_from_training.txt`, so no adapter arm in the thesis trained on it. Its
reference-count mix is 152 / 120 / 30, identical in proportion to test.

The comparator is `scores/screen__baseline.jsonl`, already generated and scored at the same
sampler settings, and `scores/screen__baseline__rerun.jsonl` supplies a measured
identical-config noise floor on the same rows.

Using this split rather than test leaves the reported evaluation set untrained on.

Written to Drive once and reused by the training and generation notebooks.

In [3]:
import yaml
from omegaconf import OmegaConf

CTRL = f"{DRIVE}/lora"
os.makedirs(f"{CTRL}/jsonls", exist_ok=True)
MEM_JSONL = f"{CTRL}/jsonls/memorize_302.jsonl"

screen = [json.loads(l) for l in open("/content/screen.jsonl", encoding="utf-8")]
test_t = {json.loads(l)["output_image"] for l in open("/content/test.jsonl", encoding="utf-8")}
val_t  = {json.loads(l)["output_image"] for l in open("/content/val.jsonl",  encoding="utf-8")}
scr_t  = {r["output_image"] for r in screen}

assert len(screen) == 302, f"expected 302 screen rows, got {len(screen)}"
assert len(scr_t) == 62, f"expected 62 targets, got {len(scr_t)}"
assert not (scr_t & test_t), "screen leaks into test"
assert not (scr_t & val_t), "screen leaks into val"

def convert(r):
    return {"task_type": "ic",
            "instruction": r["instruction"],
            "input_images": [os.path.join(IMGS, p) for p in r["input_images"]],
            "output_image": os.path.join(IMGS, r["output_image"])}

if not os.path.exists(MEM_JSONL):
    with open(MEM_JSONL, "w", encoding="utf-8") as f:
        for r in screen:
            f.write(json.dumps(convert(r), ensure_ascii=False) + "\n")
    print("wrote", os.path.basename(MEM_JSONL), len(screen), "rows")
else:
    print("jsonl already present")

# The Drive path contains an apostrophe, which closes a single-quoted YAML scalar early.
# json.dumps emits a correctly escaped double-quoted scalar, which YAML accepts verbatim.
def _cfg(target):
    return 'data:\n  - \n    path: %s\n    type: "ic"\n    ratio: !!float 1\n' % json.dumps(target)

# rewritten only when the content differs, so concurrent runs on the same Drive folder
# never truncate a file another run is reading
def _write_if_changed(path, body):
    try:
        if open(path, encoding="utf-8").read() == body:
            return False
    except FileNotFoundError:
        pass
    open(path, "w", encoding="utf-8").write(body)
    return True

for _name, _target in [("memorize_ic.yml",  MEM_JSONL),
                       ("memorize_mix.yml", f"{CTRL}/memorize_ic.yml")]:
    _p = f"{CTRL}/{_name}"
    _changed = _write_if_changed(_p, _cfg(_target))
    _parsed = yaml.safe_load(open(_p, encoding="utf-8"))["data"][0]["path"]
    assert _parsed == _target, f"{_name} parsed to {_parsed}"
    print(f"{_name}: {'written' if _changed else 'unchanged'}, parses to {os.path.basename(_target)}")

rows = [json.loads(l) for l in open(MEM_JSONL, encoding="utf-8")]
missing = [p for r in rows for p in r["input_images"] + [r["output_image"]] if not os.path.exists(p)]
assert not missing, f"{len(missing)} missing images, first: {missing[:2]}"
print(f"{len(rows)} rows, all images present, {len(rows)/16:.1f} steps per epoch at global batch 16")

wrote memorize_302.jsonl 302 rows
memorize_ic.yml: written, parses to memorize_302.jsonl
memorize_mix.yml: written, parses to memorize_ic.yml
302 rows, all images present, 18.9 steps per epoch at global batch 16


## 4. Patch train.py

Three edits. `FIX_ACCUM_NORM` matches the gradient scale every other arm in the thesis ran at.
The clip call is rebound so the pre-clip total norm is readable: reading `.grad` after that call
returns the clipped value and censors the column at `max_grad_norm`. The third prints the LoRA
`B` norm every five steps. `B` is zero-initialised, so its norm is absolute movement from init
and stays at zero for a rate that does nothing.

Idempotent. `git checkout -- train.py` reverts, in section 7.

In [4]:
p = f"{REPO}/train.py"
src = open(p, encoding="utf-8").read()

# The "* accelerator.gradient_state.num_steps" factor in the loss normalisation is a no-op on
# the reference 8-GPU setup, where num_steps is 1; with accumulation 8 on one GPU it becomes an
# active multiplier and inflates the gradient by exactly the accumulation factor. True
# reproduces the reference gradient scale, as in every other arm.
FIX_ACCUM_NORM = True

A3     = "                    accelerator.clip_grad_norm_(trainable_params, args.train.max_grad_norm)\n"
A3_NEW = "                    _gnorm = accelerator.clip_grad_norm_(trainable_params, args.train.max_grad_norm)\n"
A4     = "                loss = (loss * accelerator.gradient_state.num_steps * accelerator.num_processes) / num_tokens_in_batch\n"
A4_NEW = "                loss = (loss * accelerator.num_processes) / num_tokens_in_batch\n"
B1     = "                global_step += 1\n"
B1_NEW = B1 + '''                if accelerator.is_main_process and global_step % 5 == 0:
                    try:
                        _bn = sum(float((_bp.detach().float() ** 2).sum())
                                  for _bk, _bp in accelerator.unwrap_model(model).named_parameters()
                                  if "lora_B" in _bk) ** 0.5
                    except Exception:
                        _bn = float("nan")
                    print(f"[probe] step {global_step} b_norm {_bn:.6e} "
                          f"grad_norm {float(_gnorm):.4e}", flush=True)
'''

if "[probe] step" not in src:
    assert src.count(A3) == 1, "clip anchor moved"
    assert src.count(A4) == 1, "loss-normalisation anchor moved"
    assert src.count(B1) == 1, "global_step anchor moved"
    src = src.replace(A3, A3_NEW).replace(B1, B1_NEW)
    if FIX_ACCUM_NORM:
        src = src.replace(A4, A4_NEW)
    open(p, "w", encoding="utf-8").write(src)

now = open(p, encoding="utf-8").read()
print("  pre-clip norm capture :", A3_NEW.strip() in now)
print("  b_norm probe          :", "[probe] step" in now)
print("  accum-norm fix        :", A4_NEW.strip() in now, f"(FIX_ACCUM_NORM={FIX_ACCUM_NORM})")

  pre-clip norm capture : True
  b_norm probe          : True
  accum-norm fix        : True (FIX_ACCUM_NORM=True)


## 5. Sweep

Three rates, 20 steps each, `warmup_t: 5`, so movement is read over the 15 post-warmup steps.
`checkpointing_steps` is set past the step budget, so nothing is written to Drive.

Read the training loss (does it fall), the `B` norm (is the adapter moving at all) and the
pre-clip gradient norm (is the update censored by clipping).

In [5]:
import time, re, statistics as st

RATES     = [1e-5, 1e-4, 3e-4]
LORA_RANK = 8
STEPS     = 20

base = OmegaConf.load(f"{REPO}/options/ft_lora.yml")
EXP  = f"{CTRL}/experiments"
os.makedirs(EXP, exist_ok=True)

# train.py writes to <repo>/experiments/<name>; symlink that to Drive so logs survive the session
link = f"{REPO}/experiments"
if os.path.islink(link) or os.path.exists(link):
    if not (os.path.islink(link) and os.readlink(link) == EXP):
        os.unlink(link) if os.path.islink(link) else shutil.rmtree(link)
        os.symlink(EXP, link)
else:
    os.symlink(EXP, link)
print("experiments ->", os.readlink(link))

LAUNCH = ["accelerate", "launch",
          "--num_processes=1", "--num_machines=1", "--machine_rank=0", "--main_process_port=29500",
          "--use_fsdp", "--fsdp_offload_params", "false",
          "--fsdp_sharding_strategy", "HYBRID_SHARD_ZERO2",
          "--fsdp_auto_wrap_policy", "TRANSFORMER_BASED_WRAP",
          "--fsdp_transformer_layer_cls_to_wrap", "OmniGen2TransformerBlock",
          "--fsdp_state_dict_type", "FULL_STATE_DICT",
          "--fsdp_forward_prefetch", "false",
          "--fsdp_use_orig_params", "True",
          "--fsdp_cpu_ram_efficient_loading", "false",
          "--fsdp_sync_module_states", "True"]

RESULTS = {}
for lr in RATES:
    c = OmegaConf.load(f"{REPO}/options/ft_lora.yml")
    c.name = f"probe_lr{lr:g}"
    c.data.data_path = f"{CTRL}/memorize_mix.yml"
    c.data.max_output_pixels = 262144          # 512 * 512
    c.data.max_input_pixels = [262144] * 4
    c.data.max_side_length = 512
    c.train.gradient_accumulation_steps = 8
    c.train.lora_rank = LORA_RANK
    c.train.learning_rate = lr
    c.train.warmup_t = 5
    c.train.max_train_steps = STEPS
    c.logger.log_with = ["tensorboard"]
    c.logger.checkpointing_steps = STEPS * 100
    CFG = f"{REPO}/options/probe_lr{lr:g}.yml"
    OmegaConf.save(c, CFG)

    print(f"\n{'='*70}\nlr = {lr:g}\n{'='*70}", flush=True)
    t0 = time.time()
    r = subprocess.run(LAUNCH + ["train.py", "--config", CFG], cwd=REPO,
                       capture_output=True, text=True,
                       env={**os.environ, "WANDB_DISABLED": "true"})
    out = r.stdout + r.stderr
    RESULTS[lr] = dict(
        rc=r.returncode,
        secs=time.time() - t0,
        bn=[float(m) for m in re.findall(r"\[probe\] step \d+ b_norm (\S+)", out)],
        gn=[float(m) for m in re.findall(r"\[probe\] step \d+ b_norm \S+ grad_norm (\S+)", out)],
        loss=[float(m) for m in re.findall(r"loss=([\d.]+)", out)],
        tail=out[-1500:])
    print(f"rc={r.returncode}  {len(RESULTS[lr]['bn'])} probe points  {(time.time()-t0)/60:.1f} min")
    if r.returncode != 0:
        print(out[-2000:])
        break

experiments -> /content/drive/MyDrive/Colab Notebooks/My Master's Thesis/Data/lora/experiments

lr = 1e-05
rc=0  4 probe points  7.5 min

lr = 0.0001
rc=0  4 probe points  5.9 min

lr = 0.0003
rc=0  4 probe points  6.0 min


## 6. Result

`B` starts at zero, so `b_norm` is absolute movement from init rather than a ratio, read against
the trained run: `lora_r8` reaches 0.0944 at step 500, 0.2688 at the selected step 1000 and
0.9189 at step 4000.

Selection is on `b_norm`. The loss column is uninformative over 20 steps: the diffusion objective
is a velocity MSE at a lognormally sampled timestep, and the per-batch spread across timesteps
swamps any optimisation signal at this horizon, which is why all three rates report the same
trajectory from the same seed and data order.

In [6]:
print(f"{'lr':>8s} {'min':>7s} {'b_norm@20':>12s} {'grad norm':>12s} {'loss first->last':>22s} {'trend':>9s}")
for lr, d in sorted(RESULTS.items()):
    bn = f"{d['bn'][-1]:.4e}" if d["bn"] else "-"
    gn = f"{d['gn'][-1]:.3e}" if d["gn"] else "-"
    if len(d["loss"]) > 5:
        head, tail = st.mean(d["loss"][:3]), st.mean(d["loss"][-3:])
        ls = f"{head:.4f} -> {tail:.4f}"
        tr = "falling" if tail < head else "rising"
    else:
        ls, tr = "-", "-"
    print(f"{lr:>8g} {d['secs']/60:>7.1f} {bn:>12s} {gn:>12s} {ls:>22s} {tr:>9s}")

print("\nreading it:")
print("  b_norm 0 or nan          -> the adapter is not in the graph; the section 4 patch is wrong")
print("  b_norm ~0 with flat loss -> rate too low; the control would be undertrained by construction")
print("  loss rising or nan       -> rate too high")
print("  loss falling             -> usable; take the highest such rate")
print(f"  clipping active at max_grad_norm={base.train.max_grad_norm}:",
      {f"{lr:g}": (max(d["gn"]) > base.train.max_grad_norm if d["gn"] else None)
       for lr, d in sorted(RESULTS.items())})
print(f"\n1000 steps at the measured 15.41 s/step: {1000*15.41/3600:.1f} h "
      f"= {1000*15.41/3600*11.77:.0f} units")

      lr     min    b_norm@20    grad norm       loss first->last     trend
   1e-05     7.5   1.0076e-01    2.054e-02       0.0420 -> 0.0550    rising
  0.0001     5.9   9.2088e-01    1.738e-02       0.0420 -> 0.0548    rising
  0.0003     6.0   2.3192e+00    1.696e-02       0.0420 -> 0.0547    rising

reading it:
  b_norm 0 or nan          -> the adapter is not in the graph; the section 4 patch is wrong
  b_norm ~0 with flat loss -> rate too low; the control would be undertrained by construction
  loss rising or nan       -> rate too high
  loss falling             -> usable; take the highest such rate
  clipping active at max_grad_norm=1: {'1e-05': False, '0.0001': False, '0.0003': False}

1000 steps at the measured 15.41 s/step: 4.3 h = 50 units


## 7. Revert the patch

Leaves the repo clean so the next session re-clones or reuses an unmodified `train.py`.

In [7]:
subprocess.run(["git", "-C", REPO, "checkout", "--", "train.py"], check=True)
src = open(f"{REPO}/train.py", encoding="utf-8").read()
assert "[probe] step" not in src and "_gnorm" not in src
assert A4 in src, "loss normalisation not restored"
print("train.py reverted to", COMMIT[:7])

train.py reverted to 18e6f9d
